# Track A · Colab 실행 노트북

VS Code에서 `Select Kernel → Colab`으로 런타임을 붙인 뒤 위에서부터 실행한다.
코드는 GitHub 레포에서 받고, 로그·체크포인트는 Drive에 남긴다.

## 0. 런타임 확인

In [ ]:
import torch, subprocess
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                         capture_output=True, text=True).stdout)

## 1. 레포 받기

private 레포면 Colab 왼쪽 🔑 Secrets에 `GH_TOKEN`(fine-grained PAT, Contents: Read)을 저장해 둔다.
토큰을 노트북에 직접 적지 말 것 — 커밋되면 GitHub가 즉시 폐기한다.

In [ ]:
import os, subprocess
REPO, DIR = 'simdongsub/Deep_learning', 'Deep_learning'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.isdir(DIR):
    subprocess.run(['git','clone',url,DIR], check=True)
else:
    subprocess.run(['git','pull'], cwd=DIR, check=True)
os.chdir(DIR)
print(os.getcwd())
print(subprocess.run(['git','log','-1','--oneline'], capture_output=True, text=True).stdout)

## 2. Drive 마운트 (로그·체크포인트 저장소)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/trackA/runs'
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

## 3. 스모크 테스트

본 실험 전에 파이프라인만 확인한다 (10% 데이터 · 2 에폭 · 2~3분).
여섯 설정이 모두 에러 없이 도는지, 파라미터 수가 예상과 맞는지 본다.

In [ ]:
!python -m scripts.param_match --target configs/f4.yaml

In [ ]:
for cfg in ['b0','b0plus','f1','f2','f3','f4']:
    print('='*60, cfg)
    !python -m src.train --config configs/{cfg}.yaml --seed 0 --fraction 0.1 --epochs 2 --out /content/smoke

## 4. 베이스라인 재현 (슬라이드 14 · 단계 02)

목표: 테스트 오차율 8.75% ±1%p. 100 에폭 · 100% 데이터.

In [ ]:
!python -m src.train --config configs/b0.yaml --seed 0 --lr 0.1 --fraction 1.0 --out {OUT}

## 5. 본 실험 배치 실행

세션이 끊겨도 같은 셀을 다시 실행하면 `ckpt_last.pt`에서 이어서 학습한다.
한 번에 다 돌리지 말고 몇 개씩 끊어 돌리는 편이 안전하다.

In [ ]:
!python -m scripts.make_jobs --stage core --lr 0.1 --out {OUT} > jobs_core.sh
!head -5 jobs_core.sh && wc -l jobs_core.sh

In [ ]:
# 예: 앞에서 6개만 실행 (숫자를 바꿔 가며 나눠 돌린다)
!sed -n '3,8p' jobs_core.sh > jobs_part.sh && bash jobs_part.sh

## 6. 결과 모으기

In [ ]:
!python -m scripts.collect_results --runs {OUT} --out /content/drive/MyDrive/trackA/results